# rag-template: замеры качества

Ноутбук считает все числа для README:

1. поиск на документации FastAPI (42 вопроса + 42 перефраза) — 6 конфигураций;
2. поиск на открытых бенчмарках: RuBQ (русский) и SciFact (английский);
3. семантический кэш: как часто перефраз получает ответ из кэша и как часто другой вопрос получает чужой ответ;
4. ответы целиком: LLM-судья проверяет правильность, отсутствие выдумок и отказы.

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**.
Для шага 4 нужен ключ Groq: значок ключа слева → Add new secret → имя `GROQ_API_KEY`, включить доступ для ноутбука.

Время: шаги 1–3 ≈ 20–30 минут, шаг 4 ≈ 15 минут.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/sofetizm/rag-template.git
%cd rag-template
!pip install -q -e ".[eval]"

Запускаем Qdrant: на больших коллекциях сервер ищет в десятки раз быстрее встроенного режима.

In [ ]:
import os
import subprocess
import time
import urllib.request

if not os.path.exists("/content/qdrant"):
    !curl -sSL https://github.com/qdrant/qdrant/releases/download/v1.20.0/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content
os.makedirs("/content/qdrant_data", exist_ok=True)
qdrant = subprocess.Popen(
    ["/content/qdrant"], cwd="/content/qdrant_data", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
for _ in range(30):
    try:
        print(urllib.request.urlopen("http://localhost:6333").read()[:80])
        break
    except Exception:
        time.sleep(1)

os.environ["RAG_QDRANT_URL"] = "http://localhost:6333"
os.environ["RAG_DEVICE"] = "cuda"

## 1. Поиск по документации FastAPI

In [ ]:
!rag ingest data/demo/fastapi-ru --base-url https://fastapi.tiangolo.com/ru/ --rebuild
!rag eval retrieval --out results/golden_retrieval.json

## 2. Открытые бенчмарки

In [ ]:
!rag eval benchmark --dataset mteb/RuBQRetrieval --out results/bench_rubq.json
!rag eval benchmark --dataset mteb/scifact --out results/bench_scifact.json

## 3. Семантический кэш

In [ ]:
!rag eval cache --out results/cache.json

## 4. Ответы целиком

Если Groq ответит, что дневной лимит исчерпан, просто запустите ячейку ещё раз на следующий день: готовые ответы сохраняются и не пересчитываются.

In [ ]:
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
!rag eval answers --out results/answers.jsonl --summary results/answers_summary.json

## Отчёт и скачивание

In [ ]:
!python scripts/make_report.py
from IPython.display import Markdown, display

display(Markdown(open("results/RESULTS.md", encoding="utf-8").read()))
!zip -qr results.zip results
from google.colab import files

files.download("results.zip")